In [ ]:
# EN3160 A02 - Q1: LoG blob detection in scale-space (sunflower field)
# ---------------------------------------------------------------
# Requirements: pip install opencv-python matplotlib numpy
import os, math, csv
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

# ---------- config ----------
# Paths: assignment path first, then fallback to uploaded file if present
CANDIDATE_PATHS = [
    "images/the_berry_farms_sunflower_field.jpeg",
    "/mnt/data/the_berry_farms_sunflower_field.jpeg",  # fallback in this environment
    "the_berry_farms_sunflower_field.jpeg"
]
OUTDIR = "outputs/q1"
os.makedirs(OUTDIR, exist_ok=True)

# Scale-space parameters (as described in the write-up)
SIGMA0 = 1.6
K = 1.2
NUM_SCALES = 15
THRESH_REL = 0.02      # relative to global max of scale-normalized LoG
LOCAL_MAX_KSIZE = 3    # 2D local maxima window (odd)
NMS_DIST_FACTOR = 0.5  # suppress if centers closer than 0.5*(ri + rj)

# How many largest circles to report/label on the image
TOPK_REPORT = 25

# ---------- helpers ----------
def read_image():
    img = None
    for p in CANDIDATE_PATHS:
        if os.path.exists(p):
            img = cv.imread(p, cv.IMREAD_COLOR)
            if img is not None:
                print(f"[INFO] Loaded: {p}")
                return img, p
    raise FileNotFoundError("Sunflower image not found in expected locations.")

def to_gray_float(img_bgr):
    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    return gray

def build_sigmas(sigma0, k, n):
    return [sigma0 * (k**i) for i in range(n)]

def log_response(gray, sigma):
    blurred = cv.GaussianBlur(gray, (0, 0), sigma)
    lap = cv.Laplacian(blurred.astype(np.float32), cv.CV_32F)
    return (sigma**2) * np.abs(lap)


def local_max_2d(resp, ksize=3):
    # 2D local maxima via dilation
    kernel = np.ones((ksize, ksize), np.uint8)
    resp32 = resp.astype(np.float32)
    # Use cv.dilate that works on single-channel images
    mx = cv.dilate(resp32, kernel)
    return (resp32 >= mx - 1e-12)  # boolean mask of local maxima

def nms_across_detections(dets, dist_factor=0.5):
    # dets: list of dicts with x,y,r,score
    # Sort by descending score (strongest first)
    dets = sorted(dets, key=lambda d: d["score"], reverse=True)
    kept = []
    for d in dets:
        keep = True
        for e in kept:
            dx = d["x"] - e["x"]
            dy = d["y"] - e["y"]
            dist = math.hypot(dx, dy)
            thr = dist_factor * (d["r"] + e["r"])
            if dist < thr:
                keep = False
                break
        if keep:
            kept.append(d)
    return kept

# ---------- main ----------
img_bgr, used_path = read_image()
gray = to_gray_float(img_bgr)
H, W = gray.shape

sigmas = build_sigmas(SIGMA0, K, NUM_SCALES)
print(f"[INFO] Sigma range used: [{sigmas[0]:.2f}, {sigmas[-1]:.2f}] with K={K} and {NUM_SCALES} scales")

# Build LoG scale-space
responses = []
for s in sigmas:
    resp = log_response(gray, s)
    responses.append(resp)
stack = np.stack(responses, axis=2)  # H x W x S
gmax = stack.max()
thr_abs = THRESH_REL * gmax
print(f"[INFO] Global max response = {gmax:.6f}, threshold = {thr_abs:.6f} ({THRESH_REL*100:.1f}% of max)")

# 3D non-maximum suppression (x,y,s)
candidates = []
S = len(sigmas)
for si in range(S):
    resp = stack[:,:,si]
    if resp.max() < thr_abs:
        continue
    # 2D local maxima at this scale
    locmax2d = local_max_2d(resp, ksize=LOCAL_MAX_KSIZE)
    # Compare with neighboring scales
    prev_resp = stack[:,:,si-1] if si-1 >= 0 else None
    next_resp = stack[:,:,si+1] if si+1 < S else None

    # points that are higher than prev and next (when they exist)
    if prev_resp is not None:
        locmax2d &= (resp >= prev_resp)
    if next_resp is not None:
        locmax2d &= (resp >= next_resp)

    ys, xs = np.where(locmax2d & (resp >= thr_abs))
    for y, x in zip(ys, xs):
        score = float(resp[y, x])
        sigma = sigmas[si]
        r = math.sqrt(2.0) * sigma
        candidates.append({"x": int(x), "y": int(y), "sigma": float(sigma), "r": float(r), "score": score})

print(f"[INFO] Raw candidates: {len(candidates)}")

# Non-maximum suppression across detections to de-duplicate
detections = nms_across_detections(candidates, dist_factor=NMS_DIST_FACTOR)
print(f"[INFO] Kept detections after NMS: {len(detections)}")

# Sort by radius to find the largest circles
detections_by_radius = sorted(detections, key=lambda d: d["r"], reverse=True)
largest = detections_by_radius[:TOPK_REPORT]

# ---- Save CSV with all detections
csv_path = os.path.join(OUTDIR, "q1_detections_all.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["x", "y", "radius_px", "sigma", "score"])
    for d in detections:
        writer.writerow([d["x"], d["y"], f"{d['r']:.2f}", f"{d['sigma']:.2f}", f"{d['score']:.6f}"])
print(f"[INFO] Saved CSV: {csv_path}")

# ---- Draw top-K largest on the image
overlay = img_bgr.copy()
for idx, d in enumerate(largest, 1):
    cv.circle(overlay, (d["x"], d["y"]), int(round(d["r"])), (0, 255, 0), 2)  # green circle
    cv.circle(overlay, (d["x"], d["y"]), 2, (0, 0, 255), -1)                  # red center
    cv.putText(overlay, f"#{idx}", (d["x"]+4, d["y"]-4), cv.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv.LINE_AA)

out_img = os.path.join(OUTDIR, "q1_circles_overlay.png")
cv.imwrite(out_img, overlay)
print(f"[INFO] Saved overlay: {out_img}")

# ---- Also save a max-over-scales LoG response image (for the report)
max_resp = stack.max(axis=2)
norm = (255 * (max_resp / (max_resp.max() + 1e-12))).astype(np.uint8)
heat = cv.applyColorMap(norm, cv.COLORMAP_TURBO)
out_heat = os.path.join(OUTDIR, "q1_log_max_response.png")
cv.imwrite(out_heat, heat)
print(f"[INFO] Saved LoG heatmap: {out_heat}")

# ---- Show quick previews in the notebook
plt.figure(figsize=(10,6))
plt.imshow(cv.cvtColor(overlay, cv.COLOR_BGR2RGB))
plt.title("Q1: Largest circles (LoG scale-space)")
plt.axis("off")
plt.show()

# ---- Print the report section with results
print("\n=== Q1: Largest circles (sorted by radius) ===")
print(f"Sigma range used: [{sigmas[0]:.2f}, {sigmas[-1]:.2f}] (k={K}, {NUM_SCALES} scales)")
for i, d in enumerate(largest, 1):
    print(f"{i:>2}. center=({d['x']}, {d['y']}), radius={d['r']:.2f}px, sigma={d['sigma']:.2f}, score={d['score']:.6f}")


[INFO] Loaded: the_berry_farms_sunflower_field.jpeg
[INFO] Sigma range used: [1.60, 20.54] with K=1.2 and 15 scales
[INFO] Global max response = 0.427003, threshold = 0.008540 (2.0% of max)
[INFO] Raw candidates: 65840
